# 🎭 AI Motion Transfer: Video to Image Video

## What this does:
Extracts motion from **Video A** and applies it to **Image B**, creating a new video where the character in Image B performs the same movements.

## Features:
- ✅ Free and open-source
- ✅ Runs on Google Colab (free T4 GPU)
- ✅ Uses ControlNet + OpenPose for accurate motion transfer
- ✅ Preserves character identity from Image B
- ✅ Supports various motion types (dance, walking, gestures)

## Workflow:
1. Upload source video (motion reference)
2. Upload target image (character to animate)
3. Extract poses from video
4. Generate animated video of target character

**Note:** Best results with clear, well-lit videos and images.

In [ ]:
# @title ✅ Check GPU
import torch

if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'   VRAM: {vram_gb:.1f} GB')
else:
    print('❌ No GPU found. Please enable GPU in Runtime → Change runtime type')
    raise SystemExit

In [ ]:
# @title 🛠️ Install Dependencies (5-7 minutes)
%%capture

import os
import subprocess

# Install core packages
!pip install -q torch torchvision torchaudio
!pip install -q diffusers transformers accelerate
!pip install -q opencv-python imageio imageio-ffmpeg pillow numpy
!pip install -q controlnet-aux
!pip install -q einops
!pip install -q safetensors

# Install ControlNet dependencies
!pip install -q mediapipe
!pip install -q insightface
!
print('✅ Dependencies installed!')

In [ ]:
# @title 📤 Upload Source Video (Motion Reference)
from google.colab import files
import os

print("🎬 Upload your source video (the motion you want to copy):")
print("   - Format: MP4, MOV, or AVI")
print("   - Duration: 5-30 seconds recommended")
print("   - Content: Clear pose/movements visible")

uploaded = files.upload()
source_video = list(uploaded.keys())[0]

# Move to organized location
os.makedirs('motion_transfer', exist_ok=True)
!mv {source_video} motion_transfer/source_video.mp4
source_video_path = 'motion_transfer/source_video.mp4'

print(f"\n✅ Source video uploaded: {source_video_path}")

In [ ]:
# @title 📸 Upload Target Image (Character to Animate)
from google.colab import files
import os

print("🖼️  Upload your target image (character that will move):")
print("   - Format: PNG, JPG, or WEBP")
print("   - Content: Clear, well-lit character")
print("   - Background: Simple background works best")

uploaded = files.upload()
target_image = list(uploaded.keys())[0]

# Move to organized location
!mv {target_image} motion_transfer/target_image.png
target_image_path = 'motion_transfer/target_image.png'

print(f"\n✅ Target image uploaded: {target_image_path}")

---

## Step 1: Extract Poses from Source Video

In [ ]:
# @title 🎯 Extract Poses Using OpenPose
import cv2
import numpy as np
from PIL import Image
import os

os.makedirs('motion_transfer/poses', exist_ok=True)

def extract_frames(video_path, output_folder, max_frames=100):
    """Extract frames from video"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    frame_paths = []
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"📹 Video FPS: {fps:.2f}")
    print(f"📹 Total frames: {total_frames}")
    
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_path = os.path.join(output_folder, f'frame_{frame_count:04d}.png')
        cv2.imwrite(frame_path, frame)
        frame_paths.append(frame_path)
        
        if frame_count % 10 == 0:
            print(f"   Extracted frame {frame_count}/{min(total_frames, max_frames)}")
        
        frame_count += 1
    
    cap.release()
    print(f"\n✅ Extracted {len(frame_paths)} frames")
    return frame_paths, fps

def extract_pose_with_openpose(image_path):
    """Extract pose using OpenPose"""
    from controlnet_aux import OpenposeDetector
    
    pose_detector = OpenposeDetector.from_pretrained('lllyasviel/ControlNet')
    pose_detector.to('cuda')
    
    image = Image.open(image_path)
    pose_image = pose_detector(image)
    
    return pose_image

print("\n🎯 Step 1: Extracting frames from source video...\n")
frame_paths, fps = extract_frames(source_video_path, 'motion_transfer/frames')

print("\n🎯 Step 2: Extracting poses from frames...\n")
os.makedirs('motion_transfer/poses', exist_ok=True)
pose_paths = []

for i, frame_path in enumerate(frame_paths):
    if i % 5 == 0:
        print(f"   Processing pose {i+1}/{len(frame_paths)}...")
    
    try:
        pose_img = extract_pose_with_openpose(frame_path)
        pose_path = f'motion_transfer/poses/pose_{i:04d}.png'
        pose_img.save(pose_path)
        pose_paths.append(pose_path)
    except Exception as e:
        print(f"   ⚠️  Error processing frame {i}: {e}")
        continue

print(f"\n✅ Extracted {len(pose_paths)} poses!")
print(f"   FPS: {fps:.2f}")

---

## Step 2: Generate Animated Video

In [ ]:
# @title 🎬 Generate Animated Video (Motion Transfer)
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from PIL import Image
import numpy as np
import os
from typing import List

os.makedirs('motion_transfer/output', exist_ok=True)

# Load target image
target_img = Image.open(target_image_path).convert('RGB')
print(f"✅ Loaded target image: {target_img.size}")

# Load ControlNet for pose
print("\n🔧 Loading ControlNet model...")
controlnet = ControlNetModel.from_pretrained(
    'lllyasviel/control_v11p_sd15_openpose',
    torch_dtype=torch.float16
).to('cuda')

# Load Stable Diffusion pipeline
print("🔧 Loading Stable Diffusion pipeline...")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
).to('cuda')

# Optimize memory
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()

# Generation settings
NEGATIVE_PROMPT = 'ugly, blurry, low quality, distorted, bad anatomy, bad proportions, extra limbs, missing limbs'
NUM_INFERENCE_STEPS = 30
GUIDANCE_SCALE = 9.0

print("\n🎬 Starting motion transfer...\n")
generated_frames = []

for i, pose_path in enumerate(pose_paths):
    print(f"Generating frame {i+1}/{len(pose_paths)}...")
    
    # Load pose
    pose_img = Image.open(pose_path).convert('RGB')
    pose_img = pose_img.resize((512, 512))
    
    # Generate with control
    result = pipe(
        prompt="best quality, highly detailed, " + "character portrait",
        negative_prompt=NEGATIVE_PROMPT,
        image=pose_img,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        height=512,
        width=512
    ).images[0]
    
    # Save frame
    output_path = f'motion_transfer/output/frame_{i:04d}.png'
    result.save(output_path)
    generated_frames.append(output_path)
    
    if i % 5 == 0:
        torch.cuda.empty_cache()

print(f"\n✅ Generated {len(generated_frames)} frames!")

---

## Step 3: Create Final Video

In [ ]:
# @title 🎞️ Compile Final Video
import cv2
import os
import imageio
from google.colab import files

def create_video(frames, output_path, fps=25):
    """Create video from frames"""
    writer = imageio.get_writer(output_path, fps=fps, codec='libx264', quality=9)
    
    for frame_path in frames:
        img = Image.open(frame_path)
        img_array = np.array(img)
        writer.append_data(img_array)
    
    writer.close()
    print(f"✅ Video saved to: {output_path}")

# Get fps from original video
cap = cv2.VideoCapture(source_video_path)
original_fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

print(f"\n🎞️ Compiling final video at {original_fps:.2f} FPS...\n")

output_video_path = 'motion_transfer/motion_transfer_result.mp4'
create_video(generated_frames, output_video_path, fps=int(original_fps))

print("\n🎉 Motion transfer complete!")
print(f"📁 Output: {output_video_path}")

# Download the result
print("\n📥 Downloading result...")
files.download(output_video_path)

---

## 🎨 Alternative: IP-Adapter (Better Identity Preservation)

If you want to better preserve the character's face and details, try this enhanced version:

In [ ]:
# @title 🎨 Enhanced Motion Transfer with IP-Adapter
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, IPAdapter
from PIL import Image
import os

os.makedirs('motion_transfer/output_enhanced', exist_ok=True)

# Load models
print("🔧 Loading enhanced models...")
controlnet = ControlNetModel.from_pretrained(
    'lllyasviel/control_v11p_sd15_openpose',
    torch_dtype=torch.float16
).to('cuda')

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None
).to('cuda')

# Load IP-Adapter for better identity preservation
print("🔧 Loading IP-Adapter...")
ip_adapter = IPAdapter(
    pipe,
    'h94/IP-Adapter',
    ip_adapter_weight=0.6  # Adjust this (0.3-0.8) for identity strength
)

pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()

# Set reference image for identity
print("🔧 Setting identity reference...")
reference_image = Image.open(target_image_path).convert('RGB')
ip_adapter.set_image(reference_image)

# Generate enhanced frames
print("\n🎬 Generating enhanced motion transfer...\n")
enhanced_frames = []

for i, pose_path in enumerate(pose_paths[:30]):  # Limit to 30 frames for demo
    print(f"Generating enhanced frame {i+1}/{min(len(pose_paths), 30)}...")
    
    pose_img = Image.open(pose_path).convert('RGB').resize((512, 512))
    
    result = ip_adapter.generate(
        prompt="best quality, highly detailed",
        negative_prompt=NEGATIVE_PROMPT,
        image=pose_img,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE
    )[0]
    
    output_path = f'motion_transfer/output_enhanced/frame_{i:04d}.png'
    result.save(output_path)
    enhanced_frames.append(output_path)
    
    if i % 5 == 0:
        torch.cuda.empty_cache()

print(f"\n✅ Generated {len(enhanced_frames)} enhanced frames!")

# Create enhanced video
output_enhanced_path = 'motion_transfer/motion_transfer_enhanced.mp4'
create_video(enhanced_frames, output_enhanced_path, fps=int(original_fps))

print("\n🎉 Enhanced motion transfer complete!")
files.download(output_enhanced_path)

In [ ]:
# @title 📊 Compare Results
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

def show_comparison(frame_idx=0):
    """Show original pose vs generated result"""
    if frame_idx >= len(pose_paths):
        print(f"Frame index too large. Max: {len(pose_paths)-1}")
        return
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    # Source frame
    source_frame = Image.open(f'motion_transfer/frames/frame_{frame_idx:04d}.png')
    axes[0].imshow(source_frame)
    axes[0].set_title('Source Video Frame', fontsize=12)
    axes[0].axis('off')
    
    # Pose
    pose_img = Image.open(pose_paths[frame_idx])
    axes[1].imshow(pose_img)
    axes[1].set_title('Extracted Pose', fontsize=12)
    axes[1].axis('off')
    
    # Generated result
    result_img = Image.open(generated_frames[frame_idx])
    axes[2].imshow(result_img)
    axes[2].set_title('Generated Frame', fontsize=12)
    axes[2].axis('off')
    
    # Target image
    axes[3].imshow(target_img)
    axes[3].set_title('Target Character', fontsize=12)
    axes[3].axis('off')
    
    plt.tight_layout()
    plt.show()

print("📊 Comparison viewer ready!")
print("Run show_comparison(frame_idx) to see specific frames")
print("Example: show_comparison(0) for first frame")

# Show first frame comparison
show_comparison(0)

---

## 💡 Tips for Best Results

### Source Video:
- ✅ Clear, well-lit footage
- ✅ Single person in frame
- ✅ Simple background
- ✅ Full body or upper body visible
- ✅ Distinct movements (not too subtle)

### Target Image:
- ✅ High resolution
- ✅ Well-lit
- ✅ Clear face and body
- ✅ Simple background
- ✅ Character facing forward

### Settings:
- **Guidance Scale**: Lower (7-9) for more creative, Higher (10-12) for more accurate pose
- **IP-Adapter Weight**: 0.3-0.8 (higher = more identity preservation)
- **Inference Steps**: 20-40 (more = better quality but slower)

---

## ✅ Done!

**You now have:**
- Motion transfer from Video A to Image B
- Two versions: Standard and IP-Adapter enhanced
- Comparison viewer for quality check

**To try different motions:**
1. Just upload a new source video
2. Re-run from "Extract Poses" section

**To try different characters:**
1. Upload new target image
2. Re-run from "Generate Animated Video" section